In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ChicagoTaxiPipeline").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/30 16:54:33 WARN Utils: Your hostname, ouyassine-ThinkPad-L14-Gen-1, resolves to a loopback address: 127.0.1.1; using 10.1.0.89 instead (on interface wlp3s0)
26/10/30 16:54:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/30 16:54:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
file_url = "/home/ouyassine/Documents/projects/taxi pipeline/data/raw_data/chicago_taxi_trips_2020_2021.csv"

dataset = spark.read.csv(file_url, header=True, inferSchema=True)



In [4]:
dataset.show(5)

dataset.printSchema()

+--------------------+--------------------+--------------------+-------------------+------------+----------+-------------------+--------------------+---------------------+----------------------+----+----+-----+------+----------+------------+--------------------+------------------------+-------------------------+------------------------+-------------------------+--------------------------+-------------------------+
|             trip_id|             taxi_id|trip_start_timestamp| trip_end_timestamp|trip_seconds|trip_miles|pickup_census_tract|dropoff_census_tract|pickup_community_area|dropoff_community_area|fare|tips|tolls|extras|trip_total|payment_type|             company|pickup_centroid_latitude|pickup_centroid_longitude|pickup_centroid_location|dropoff_centroid_latitude|dropoff_centroid_longitude|dropoff_centroid_location|
+--------------------+--------------------+--------------------+-------------------+------------+----------+-------------------+--------------------+---------------

In [5]:
print("Number of rows: ", dataset.count())

Number of rows:  4560000


In [6]:
print("columns: ", dataset.columns)

columns:  ['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp', 'trip_seconds', 'trip_miles', 'pickup_census_tract', 'dropoff_census_tract', 'pickup_community_area', 'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras', 'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude', 'pickup_centroid_longitude', 'pickup_centroid_location', 'dropoff_centroid_latitude', 'dropoff_centroid_longitude', 'dropoff_centroid_location']


# Data cleaning

In [10]:
from pyspark.sql.functions import col, sum

# how many null values are in each column
null_counts = dataset.select([sum(dataset[col].isNull().cast("int")).alias(col) for col in dataset.columns])
null_counts.show()

#percetange of null values in each column
total_rows = dataset.count()
null_percentage = null_counts.select([
    (col(column_name) / total_rows * 100).alias(column_name)
    for column_name in null_counts.columns
])
null_percentage.show()

+-------+-------+--------------------+------------------+------------+----------+-------------------+--------------------+---------------------+----------------------+----+----+-----+------+----------+------------+-------+------------------------+-------------------------+------------------------+-------------------------+--------------------------+-------------------------+
|trip_id|taxi_id|trip_start_timestamp|trip_end_timestamp|trip_seconds|trip_miles|pickup_census_tract|dropoff_census_tract|pickup_community_area|dropoff_community_area|fare|tips|tolls|extras|trip_total|payment_type|company|pickup_centroid_latitude|pickup_centroid_longitude|pickup_centroid_location|dropoff_centroid_latitude|dropoff_centroid_longitude|dropoff_centroid_location|
+-------+-------+--------------------+------------------+------------+----------+-------------------+--------------------+---------------------+----------------------+----+----+-----+------+----------+------------+-------+----------------------

+-------+-------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+---------------------+----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+-------+------------------------+-------------------------+------------------------+-------------------------+--------------------------+-------------------------+
|trip_id|taxi_id|trip_start_timestamp|  trip_end_timestamp|       trip_seconds|          trip_miles|pickup_census_tract|dropoff_census_tract|pickup_community_area|dropoff_community_area|                fare|                tips|               tolls|              extras|          trip_total|payment_type|company|pickup_centroid_latitude|pickup_centroid_longitude|pickup_centroid_location|dropoff_centroid_latitude|dropoff_centroid_longitude|dropoff_centroid_location|
+-------+-------+--------------------+--------------------+-----